# Safehouse Value-Add Pipeline: Lighthouse Sanctuary

## 1. Problem Framing

### Business problem
Executive and Operations leadership need to know which safehouses **systematically outperform or underperform** after accounting for **case mix** (baseline difficulty of residents), and whether **Operations** funding aligns with that case-adjusted performance.

### Explanatory (value-added) objective — not prediction
This notebook is an **explanatory / causal-inference style** analysis (multiple linear regression for interpretation), consistent with **Chapter 9**: we estimate **conditional associations** while holding other measured factors constant. We are **not** building a model to forecast future health scores for new residents (that predictive framing is **Chapter 11** and is not the goal here).

### Why raw averages mislead
Ranking safehouses by **raw average** outcome change mixes two things: (1) the facility’s contribution and (2) **who** the facility serves. A safehouse that accepts higher-risk residents can look “worse” even if it delivers strong support. Value-added style modeling adds **case-mix controls** and **safehouse indicators** so leadership can separate composition from facility-specific patterns — while remaining honest about **omitted-variable bias** and the limits of observational data.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

candidate_dirs = [Path("../../data/raw"), Path("data/raw")]
DATA_DIR = next((p for p in candidate_dirs if p.exists()), Path("../../data/raw"))
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print(f"Data dir: {DATA_DIR.resolve()} (exists={DATA_DIR.exists()})")


## 2. Data Acquisition, Preparation & Exploration

We load resident, health, safehouse, and allocation tables, then build the **resident–month spine** used in Pipeline 1. The sustained outcome is **`health_score_delta_6m`**: general health at month **T+6** minus general health at month **T**.

We also compute a numeric **`age_upon_admission_years`** from `date_of_birth` and `date_of_admission` (rather than using the human-readable string column in `residents.csv`). **Operations** funding is the sum of `amount_allocated` in `donation_allocations.csv` where `program_area == 'Operations'`.


In [ ]:
# --- Load ---
residents = pd.read_csv(DATA_DIR / "residents.csv")
health = pd.read_csv(DATA_DIR / "health_wellbeing_records.csv")
safehouses = pd.read_csv(DATA_DIR / "safehouses.csv")
donation_allocations = pd.read_csv(DATA_DIR / "donation_allocations.csv")

# Parse dates
residents["date_of_birth"] = pd.to_datetime(residents["date_of_birth"], errors="coerce")
residents["date_of_admission"] = pd.to_datetime(residents["date_of_admission"], errors="coerce")
health["record_date"] = pd.to_datetime(health["record_date"], errors="coerce")
health["month"] = health["record_date"].dt.to_period("M").dt.to_timestamp()

# Numeric age at admission (years)
residents["age_upon_admission_years"] = (
    (residents["date_of_admission"] - residents["date_of_birth"]).dt.days / 365.25
)

# Operations budget per safehouse
ops_alloc = donation_allocations.loc[donation_allocations["program_area"].eq("Operations")].copy()
ops_budget_by_sh = (
    ops_alloc.groupby("safehouse_id", as_index=False)["amount_allocated"].sum().rename(
        columns={"amount_allocated": "operations_budget_total"}
    )
)

print("Shapes:", {"residents": residents.shape, "health": health.shape, "safehouses": safehouses.shape, "donation_allocations": donation_allocations.shape})
print("\nOperations budget by safehouse (head):")
print(ops_budget_by_sh.sort_values("safehouse_id").head().to_string())


In [ ]:
# --- Resident–month spine + 6-month delta (same pattern as Pipeline 1) ---
resident_month_ranges = (
    health.groupby("resident_id")["month"].agg(month_min="min", month_max="max").dropna().reset_index()
)

spine_rows = []
for _, row in resident_month_ranges.iterrows():
    for m in pd.date_range(start=row["month_min"], end=row["month_max"], freq="MS"):
        spine_rows.append((row["resident_id"], m))

spine = pd.DataFrame(spine_rows, columns=["resident_id", "month"])

health_monthly = health[["resident_id", "month", "general_health_score"]].copy()
health_t = health_monthly.rename(columns={"general_health_score": "general_health_score_t"})
health_tp6 = health_monthly.copy()
health_tp6["month"] = health_tp6["month"] - pd.DateOffset(months=6)
health_tp6 = health_tp6.rename(columns={"general_health_score": "general_health_score_t_plus_6"})

spine = spine.merge(health_t, on=["resident_id", "month"], how="left")
spine = spine.merge(health_tp6, on=["resident_id", "month"], how="left")
spine["health_score_delta_6m"] = (
    spine["general_health_score_t_plus_6"] - spine["general_health_score_t"]
)

case_mix_cols = [
    "resident_id",
    "safehouse_id",
    "initial_risk_level",
    "case_category",
    "is_pwd",
    "family_is_4ps",
    "age_upon_admission_years",
]
model_df = spine.merge(residents[case_mix_cols], on="resident_id", how="left")

print("Spine rows:", len(spine))
print("Rows with non-null 6-month delta:", model_df["health_score_delta_6m"].notna().sum())
model_df.head()


In [ ]:
# --- Exploration: naive average delta by safehouse (misleading if case mix differs) ---
eda = model_df.dropna(subset=["health_score_delta_6m", "safehouse_id"]).copy()
naive_by_sh = (
    eda.groupby("safehouse_id", as_index=False)["health_score_delta_6m"]
    .mean()
    .sort_values("health_score_delta_6m")
)
naive_by_sh = naive_by_sh.merge(safehouses[["safehouse_id", "name", "safehouse_code"]], on="safehouse_id", how="left")

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=naive_by_sh, x="safehouse_id", y="health_score_delta_6m", color="steelblue", ax=ax)
ax.set_title("Naive comparison: mean 6-month health score delta by safehouse")
ax.set_xlabel("safehouse_id")
ax.set_ylabel("Mean health_score_delta_6m")
plt.tight_layout()
plt.show()

# Outcome distribution (Chapter 10: skew informs residual behavior)
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(eda["health_score_delta_6m"], bins=20, kde=True, color="teal", ax=ax)
ax.set_title("Distribution of health_score_delta_6m (non-null rows)")
ax.set_xlabel("health_score_delta_6m")
plt.tight_layout()
plt.show()


## 3. Modeling & Feature Selection

We estimate a **multiple linear regression (OLS)** model (**Chapter 9**): the outcome is `health_score_delta_6m`. Predictors include **case-mix controls** and **safehouse fixed effects** implemented with **dummy variables** (`drop_first=True`), so each coefficient compares that safehouse to an **omitted reference** safehouse (the dropped `safehouse_id` dummy, here **safehouse 1**) after conditioning on measured case mix. Categorical case-mix dummies omit **Critical** (risk) and **Abandoned** (case category) as reference levels.

**Feature selection** here is **domain-driven** (per instructions): `initial_risk_level`, `case_category`, `is_pwd`, `family_is_4ps`, and `age_upon_admission_years`, plus `safehouse_id` dummies.

After fitting, we report **Chapter 10** diagnostics focused on explanatory modeling: **VIF** on the **case-mix** design (excluding many safehouse indicators, which can destabilize VIF), **residuals vs fitted**, and a **Q–Q** plot for residual normality as a *signal*, not a mechanical pass/fail.


In [ ]:
reg = model_df.dropna(subset=["health_score_delta_6m"]).copy()

# Encode features for statsmodels OLS
X_cat = pd.get_dummies(reg[["initial_risk_level", "case_category"]], drop_first=True)
X_num = pd.DataFrame(
    {
        "is_pwd": reg["is_pwd"].astype(int),
        "family_is_4ps": reg["family_is_4ps"].astype(int),
        "age_upon_admission_years": reg["age_upon_admission_years"].astype(float),
    }
)
X_sh = pd.get_dummies(reg["safehouse_id"], prefix="safehouse_id", drop_first=True)

X = pd.concat([X_num, X_cat, X_sh], axis=1)
X = sm.add_constant(X).astype(float)
y = reg["health_score_delta_6m"].astype(float)

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())


In [ ]:
# Chapter 9: descriptive in-sample error on the regression sample
y_hat = ols_model.fittedvalues
mae = np.mean(np.abs(y - y_hat))
rmse = np.sqrt(np.mean((y - y_hat) ** 2))
print(f"In-sample MAE: {mae:.4f} | In-sample RMSE: {rmse:.4f}")
print(f"R-squared: {ols_model.rsquared:.4f} | Adj. R-squared: {ols_model.rsquared_adj:.4f}")


In [ ]:
# Chapter 10: VIF for case-mix matrix only (safehouse FE dummies omitted — many categories)
X_case = pd.concat([X_num, X_cat], axis=1).astype(float)
vif_rows = []
for i in range(X_case.shape[1]):
    vif_rows.append(
        {"feature": X_case.columns[i], "VIF": variance_inflation_factor(X_case.values, i)}
    )
vif_df = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
print("VIF (case-mix predictors, no intercept):")
print(vif_df.to_string())

# Residual diagnostics
resid = ols_model.resid
fitted = ols_model.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(fitted, resid, alpha=0.7)
axes[0].axhline(0, color="black", lw=1)
axes[0].set_title("Residuals vs fitted")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")

sm.qqplot(resid, line="45", ax=axes[1])
axes[1].set_title("Normal Q-Q (residuals)")
plt.tight_layout()
plt.show()


In [ ]:
# Safehouse value-add table (coefficients + p-values)
param_names = [p for p in ols_model.params.index if str(p).startswith("safehouse_id_")]
safehouse_effects = (
    pd.DataFrame(
        {
            "term": param_names,
            "coef": ols_model.params[param_names].values,
            "pvalue": ols_model.pvalues[param_names].values,
        }
    )
    .assign(
        safehouse_id=lambda d: d["term"].str.replace("safehouse_id_", "", regex=False).astype(int)
    )
    .merge(safehouses[["safehouse_id", "name", "safehouse_code"]], on="safehouse_id", how="left")
    .sort_values("coef")
)

# Reference safehouse (dropped dummy): coefficient 0 by construction
ref_id = int(sorted(reg["safehouse_id"].dropna().unique())[0])
ref_row = pd.DataFrame(
    [
        {
            "term": f"(reference: safehouse_id={ref_id})",
            "coef": 0.0,
            "pvalue": np.nan,
            "safehouse_id": ref_id,
            "name": safehouses.loc[safehouses["safehouse_id"].eq(ref_id), "name"].iloc[0],
            "safehouse_code": safehouses.loc[safehouses["safehouse_id"].eq(ref_id), "safehouse_code"].iloc[0],
        }
    ]
)

safehouse_effects_all = pd.concat([ref_row, safehouse_effects], ignore_index=True)
safehouse_effects_all


## 4. Evaluation & Interpretation

We relate **estimated safehouse value-add** (safehouse coefficients, including the reference category at **0**) to **total Operations budget** per safehouse, then quantify association with **Pearson** and **Spearman** correlation.

This section is **evaluative** for the business question (“does funding follow true value-add?”), not a claim that the regression’s predictive accuracy is the primary success metric (**Chapter 11** differs).


In [ ]:
# Attach budget + coefs for plotting
coef_map = dict(zip(safehouse_effects_all["safehouse_id"], safehouse_effects_all["coef"]))
plot_df = ops_budget_by_sh.copy()
plot_df["value_add_coef"] = plot_df["safehouse_id"].map(coef_map)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(plot_df["value_add_coef"], plot_df["operations_budget_total"], s=80, alpha=0.85)
for _, r in plot_df.iterrows():
    ax.annotate(
        str(int(r["safehouse_id"])),
        (r["value_add_coef"], r["operations_budget_total"]),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=9,
    )
ax.set_xlabel("Safehouse value-add coefficient (vs reference safehouse)")
ax.set_ylabel("Total Operations budget allocated")
ax.set_title("Value-add vs Operations budget (safehouse level)")
plt.tight_layout()
plt.show()

mask = plot_df["value_add_coef"].notna() & plot_df["operations_budget_total"].notna()
pearson_r, pearson_p = stats.pearsonr(plot_df.loc[mask, "value_add_coef"], plot_df.loc[mask, "operations_budget_total"])
spearman_r, spearman_p = stats.spearmanr(plot_df.loc[mask, "value_add_coef"], plot_df.loc[mask, "operations_budget_total"])

print(f"Pearson r = {pearson_r:.3f} (p = {pearson_p:.4f})")
print(f"Spearman rho = {spearman_r:.3f} (p = {spearman_p:.4f})")


### Interpretation (quadrants)

Use the scatter to locate **high vs low value-add** (horizontal axis) and **high vs low Operations funding** (vertical axis). Facilities in **high value-add / low funding** can signal efficiency (or measurement issues); **low value-add / high funding** can motivate audits or operational support. Correlation summarizes **linear / monotonic alignment** between the case-mix-adjusted ranking proxy and budget — it does **not** prove that funding *caused* outcomes (see Section 5).


## 5. Causal and Relationship Analysis

### Case-mix coefficients (why adjustment matters)
Inspect the OLS table for `initial_risk_level` and `case_category` dummy coefficients. With `drop_first=True`, the **omitted reference** levels are **`initial_risk_level == Critical`** and **`case_category == Abandoned`**. Coefficients on the remaining levels are **differences relative to those baselines**. If case-mix indicators shift the outcome in plausible directions, that supports the claim that **composition differs across safehouses** and naive facility rankings can be misleading.

### Chapter 10 diagnostics and cautious language
If **VIF** indicates multicollinearity among case-mix predictors, or **heteroscedasticity** appears in residuals vs fitted, standard errors and p-values should be interpreted cautiously. Diagnostics are **signals** guiding judgment, not automatic disqualification.

### Omitted-variable bias (OVB)
Even with measured case mix, unobserved factors may still confound the **safehouse effect**: e.g., local community resources, staffing stability/tenure, referral network quality, unmeasured trauma severity, and policy changes over time.

### Budget vs value-add correlation is not causal
A positive correlation between **Operations** totals and estimated value-add **does not** imply that increasing the budget *causes* better outcomes. It is at best an **allocative efficiency** diagnostic: whether funding tracks measured performance after case-mix adjustment.


## 6. Deployment Notes

### Resource allocation dashboard (Executive / Finance)
A practical deployment is a **web dashboard** that plots each safehouse on the same two axes used above: **case-mix-adjusted value-add** (from the fixed-effects regression) vs **total Operations allocation**. The board can be required to **justify** continued high funding for low value-add facilities, or to investigate **why** some facilities achieve strong outcomes with modest Operations inflows.

### Governance workflow
- **Quarterly refresh**: re-run this notebook when new monthly health records and allocations arrive.
- **Audit triggers**: automatic flags for **low value-add / high funding** quadrants; optional drill-down into staffing and incident metrics (outside this notebook’s scope).

This turns the analysis from a one-off chart into an ongoing **resource allocation** conversation tied to transparent modeling assumptions and limitations.
